# 02 — SQL RAG

*Level 3 — Modular RAG*

## Objective
Answer natural-language questions against a **real relational database** — the open-source (MIT-licensed) Chinook sample database — via text-to-SQL, with guardrails validating every query before it runs.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "sql-rag"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from schema import core_schema
print(core_schema()[:800])


CREATE TABLE [Album]
(
    [AlbumId] INTEGER  NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    CONSTRAINT [PK_Album] PRIMARY KEY  ([AlbumId]),
    FOREIGN KEY ([ArtistId]) REFERENCES [Artist] ([ArtistId]) 
		ON DELETE NO ACTION ON UPDATE NO ACTION
)

CREATE TABLE [Artist]
(
    [ArtistId] INTEGER  NOT NULL,
    [Name] NVARCHAR(120),
    CONSTRAINT [PK_Artist] PRIMARY KEY  ([ArtistId])
)

CREATE TABLE [Customer]
(
    [CustomerId] INTEGER  NOT NULL,
    [FirstName] NVARCHAR(40)  NOT NULL,
    [LastName] NVARCHAR(20)  NOT NULL,
    [Company] NVARCHAR(80),
    [Address] NVARCHAR(70),
    [City] NVARCHAR(40),
    [State] NVARCHAR(40),
    [Country] NVARCHAR(40),
    [PostalCode] NVARCHAR(10),
    [Phone] NVARCHAR(24),
    [Fax] NVARCHAR(24),
    [Email] NVAR


## Text-to-SQL on real questions


In [3]:
from text_to_sql import answer_from_sql

questions = [
    "How many tracks are there in total?",
    "List the top 5 genres by number of tracks.",
    "Which artist has the most albums?",
    "What is the total number of customers?",
]
for q in questions:
    result = answer_from_sql(q)
    print(f"Q: {q}")
    print(f"  SQL: {result['sql']}")
    print(f"  Rows: {result['rows'][:5]}")
    print()


Q: How many tracks are there in total?
  SQL: SELECT COUNT(T1.TrackId) FROM Track AS T1 INNER JOIN Album AS T2 ON T1.AlbumId = T2.AlbumId
  Rows: [{'COUNT(T1.TrackId)': 3503}]



Q: List the top 5 genres by number of tracks.
  SQL: SELECT T1.Name FROM Genre AS T1 INNER JOIN Track AS T2 ON T1.GenreId = T2.GenreId GROUP BY T1.Name ORDER BY COUNT(T2.TrackId) DESC LIMIT 5
  Rows: [{'Name': 'Rock'}, {'Name': 'Latin'}, {'Name': 'Metal'}, {'Name': 'Alternative & Punk'}, {'Name': 'Jazz'}]



Q: Which artist has the most albums?
  SQL: SELECT T1.Name FROM Artist AS T1 INNER JOIN Album AS T2 ON T1.ArtistId = T2.ArtistId GROUP BY T1.Name ORDER BY COUNT(T2.AlbumId) DESC LIMIT 1
  Rows: [{'Name': 'Iron Maiden'}]



Q: What is the total number of customers?
  SQL: SELECT COUNT([CustomerId]) FROM [Customer] LIMIT 1
  Rows: [{'COUNT([CustomerId])': 59}]



## Guardrails in action — a deliberately unsafe query


In [4]:
from sql_guardrails import UnsafeQueryError, validate_sql

unsafe_queries = [
    "DROP TABLE Track",
    "UPDATE Track SET Name = 'hacked'",
    "SELECT * FROM Track; DROP TABLE Track",
    "SELECT * FROM Track",  # safe, but missing LIMIT
]
for sql in unsafe_queries:
    try:
        safe = validate_sql(sql)
        print(f"ALLOWED (possibly modified): {sql!r} -> {safe!r}")
    except UnsafeQueryError as e:
        print(f"BLOCKED: {sql!r} -> {e}")


BLOCKED: 'DROP TABLE Track' -> Only SELECT statements are allowed.
BLOCKED: "UPDATE Track SET Name = 'hacked'" -> Only SELECT statements are allowed.
BLOCKED: 'SELECT * FROM Track; DROP TABLE Track' -> Multi-statement SQL is not allowed.
ALLOWED (possibly modified): 'SELECT * FROM Track' -> 'SELECT * FROM Track LIMIT 100'


## What I observed

The LLM correctly generated working SQL for every real question above, including a `GROUP BY`/`ORDER BY`/`LIMIT` query for "top 5 genres" and a `JOIN` to find the artist with the most albums (Iron Maiden, in the real Chinook data). Every genuinely destructive query was blocked before it ever touched the database — the guardrail runs *before* execution, not after.

## Next

[03 — Graph RAG](./03_graph_rag.ipynb)
